# 05 — Follower Count Regression

This cross-sectional model describes association, not future growth or causality.

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
fingerprint_paths = [PROJECT_ROOT / "config" / "config.yaml", project_path("shops")]
for key in ("products", "reviews"):
    fingerprint_paths.extend(sorted(project_path(key).glob("shop_*.json")))
seller_candidate = project_path("seller_metrics")
for candidate in (seller_candidate, seller_candidate.with_suffix(".xlsx")):
    if candidate.exists(): fingerprint_paths.append(candidate)
digest = hashlib.sha256()
for path in fingerprint_paths:
    digest.update(str(path.relative_to(PROJECT_ROOT)).encode("utf-8")); digest.update(path.read_bytes())
INPUT_FINGERPRINT = digest.hexdigest()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

shops = pd.read_csv(PROCESSED / "shops_clean.csv")
auth_path = OUTPUT / "reports" / "Shop_Authenticity_Report.csv"
if not auth_path.exists(): raise FileNotFoundError("Shop_Authenticity_Report.csv is required")
data = shops.merge(pd.read_csv(auth_path)[["Shop_ID","Authentic_Review_Rate"]],on="Shop_ID",how="left",validate="one_to_one")
features = CONFIG["regression"]["features"]; target = CONFIG["regression"]["target"]
data = data.dropna(subset=[target,*features]).copy()
if len(data) < 15: raise ValueError(f"Need at least 15 complete shops; found {len(data)}")
x_train,x_test,y_train,y_test=train_test_split(data[features],data[target],test_size=max(3,int(np.ceil(len(data)*.2))),random_state=CONFIG["project"]["random_state"])
model=LinearRegression().fit(x_train,y_train); prediction=model.predict(x_test)
x_scaler,y_scaler=StandardScaler(),StandardScaler(); standardized=LinearRegression().fit(x_scaler.fit_transform(data[features]),y_scaler.fit_transform(data[[target]]).ravel())
coefficients=pd.DataFrame({"Feature":features,"Coefficient":model.coef_,"Standardized_Coefficient":standardized.coef_}); coefficients["Direction"]=np.where(coefficients["Coefficient"]>=0,"positive","negative"); coefficients["Absolute_Standardized_Coefficient"]=coefficients["Standardized_Coefficient"].abs()
metrics={"input_fingerprint":INPUT_FINGERPRINT,"MAE":mean_absolute_error(y_test,prediction),"RMSE":mean_squared_error(y_test,prediction)**.5,"R2":r2_score(y_test,prediction)}

In [ ]:
residuals=y_test-prediction
fig,axes=plt.subplots(1,2,figsize=(10,4)); sns.scatterplot(x=prediction,y=residuals,ax=axes[0]); axes[0].axhline(0,color="red"); axes[0].set(xlabel="Predicted",ylabel="Residual",title="Residuals"); sns.histplot(residuals,kde=True,ax=axes[1]); axes[1].set_title("Residual distribution"); fig.tight_layout(); fig.savefig(OUTPUT/"figures"/"regression_residuals.png",dpi=160); plt.show(); plt.close(fig)
corr=data[features].corr().abs(); high=[]
for i,a in enumerate(features):
    for b in features[i+1:]:
        if corr.loc[a,b]>=.8: high.append({"Feature_A":a,"Feature_B":b,"Absolute_Correlation":corr.loc[a,b]})
coefficients.to_csv(OUTPUT/"reports"/"Regression_Result.csv",index=False); pd.DataFrame(high).to_csv(OUTPUT/"reports"/"regression_multicollinearity_flags.csv",index=False)
(OUTPUT/"reports"/"regression_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
display(coefficients.sort_values("Absolute_Standardized_Coefficient",ascending=False)); display(pd.DataFrame([metrics])); display(pd.DataFrame(high))